[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahirtonlopes/AI-and-Robotics-Trends/blob/main/Notebooks/Demo3_Fraud_Detection.ipynb)

# Demo 3 - Demonstração - Aprendizagem Supervisionada

### Cybersecurity MBA - Artificial Intelligence & Robotics Trends (SEG)

### Algoritmo para classificação de transações em possíveis fraudes/não fraude.

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. - https://www.kaggle.com/pwnpen/payment

In [ ]:
# Import de nossas bibliotecas

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score)

In [ ]:
'''
# Upload de nosso arquivo.

from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
'''

In [ ]:
# Upload de nosso arquivo. Ref. - https://www.kaggle.com/pwnpen/payment
# Baixando nossa base de dados direto do repositório da disciplina (https://github.com/ahirtonlopes/AI-and-Robotics-Trends/) para o Google Colab

!curl -O https://raw.githubusercontent.com/ahirtonlopes/AI-and-Robotics-Trends/main/Datasets/payment_fraud.csv

In [ ]:
# Lendo dados a partir de nosso arquivo .csv
df = pd.read_csv('payment_fraud.csv')

In [ ]:
# Amostragem de cabeçalho e cinco primeiras linhas de nosso dataset
df.sample(5)

In [ ]:
# Verificando linhas e partir do índice de nosso dataset
len(df.index)

In [ ]:
# Verificando colunas de nosso dataset
df.columns

In [ ]:
# Verificando itens únicos de nossa coluna 'paymentMethod' de nosso dataset
(df['paymentMethod'].unique())

In [ ]:
# Agrupando número de itens de nossa coluna 'numItems' de nosso dataset
df.groupby('numItems').size().reset_index()

In [ ]:
# Tratando nossos dados da coluna 'pamentMethod' para formato numérico - Ref. https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html
df_one_hot = pd.get_dummies(df, columns=['paymentMethod'])

In [ ]:
df_one_hot.sample(3)

In [ ]:
# Divisão em sets de treinamento/teste (Regra de Pareto - 80/20 - Ref. https://pt.wikipedia.org/wiki/Princ%C3%ADpio_de_Pareto)
X_train, X_test, y_train, y_test = train_test_split(df_one_hot.drop('label', axis=1), df_one_hot['label'], test_size=0.2, random_state=42)

In [ ]:
len(X_train.columns)

In [ ]:
# (max_iter=1000: com 20 iterações o otimizador costuma avisar que não convergiu)
# Construção de nosso modelo usando Regressão Logística (Ref. https://edisciplinas.usp.br/pluginfile.php/3769787/mod_resource/content/1/09_RegressaoLogistica.pdf)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)

# Predição em dados de teste
y_pred = clf.predict(X_test)

In [ ]:
# Avaliação de Acurácia, Precisão e Recall de nosso modelo
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

In [ ]:
# Utilizando Árvore de Decisão
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# Avaliação de Acurácia, Precisão e Recall de nosso modelo
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

## O resultado é perfeito. Isso é bom ou é suspeito?

Os dois modelos acertam 100%. Antes de comemorar, três verificações: **(1)** comparar com um modelo trivial; **(2)** olhar a matriz de confusão; **(3)** procurar uma coluna que entregue a resposta.

> Nota: nas versões anteriores deste notebook, `accuracy_score`, `precision_score` e `recall_score` eram chamadas com os argumentos invertidos (`y_pred` antes de `y_test`), o que **troca precision e recall**. Com 100% não aparecia; com um modelo imperfeito apareceria. A ordem correta é `(y_verdadeiro, y_previsto)`.

In [ ]:
# (1) Baseline trivial: um 'modelo' que sempre responde 'normal'
import numpy as np
from sklearn.metrics import confusion_matrix

y_base = np.zeros(len(y_test), dtype=int)
print(f"Fraudes na base: {int(df['label'].sum())} de {len(df)} ({df['label'].mean():.2%})")
print(f"Fraudes no teste: {int(y_test.sum())} de {len(y_test)}")
print(f"Acurácia do 'sempre normal': {accuracy_score(y_test, y_base):.4f}  |  recall: {recall_score(y_test, y_base):.2f}")

In [ ]:
# (2) Matriz de confusão da árvore (linhas = real, colunas = previsto)
print(pd.DataFrame(confusion_matrix(y_test, y_pred), index=['real: normal', 'real: fraude'], columns=['previsto: normal', 'previsto: fraude']))

In [ ]:
# (3) Existe uma coluna que entrega a resposta? Compare as colunas por classe.
print(df.groupby('label')[['accountAgeDays', 'numItems', 'localTime', 'paymentMethodAgeDays']].agg(['min', 'max']).T)

# Uma regra de uma única pergunta:
regra = (df['accountAgeDays'] <= 1.5).astype(int)
print('\nRegra: accountAgeDays <= 1.5')
print(' acertos:', (regra == df['label']).mean())

**Leitura:** toda fraude tem `accountAgeDays = 1`, e as transações normais têm pelo menos 2. Uma única regra separa tudo. A base é sintética e fácil demais; um fraudador real envelheceria a conta. **Resultado perfeito em segurança é sinal de alerta: procure vazamento ou atalho.**

## Limiar de decisão

A regressão logística devolve uma probabilidade. O corte padrão (0,5) é só convenção.

In [ ]:
# Probabilidades e limiares (modelo de regressão logística)
from sklearn.metrics import precision_score, recall_score
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
proba = lr.predict_proba(X_test)[:, 1]

print('limiar | precision | recall | VP FP FN')
for t in (0.5, 0.3, 0.1, 0.05):
    p = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    print(f' {t:5.2f} | {precision_score(y_test, p, zero_division=0):9.3f} | {recall_score(y_test, p):6.3f} | {tp} {fp} {fn}')

## Para praticar

1. **Quebre o atalho:** remova `accountAgeDays` (`X_train.drop(columns='accountAgeDays')`) e treine de novo. O que acontece com recall e precision?
2. **Limiar:** com o problema sem atalho, varie o limiar de 0,5 a 0,05 e monte a tabela. Qual você escolheria se uma fraude perdida custa 20 vezes mais que um cliente bloqueado?
3. **Separação:** use `train_test_split(..., stratify=y)` e `cross_val_score(..., scoring='recall', cv=5)`. O resultado se mantém?
4. **Custo:** escreva em uma frase qual erro é mais caro no seu caso do trabalho final.